# MELGGen — MELG-607-64 equidistribution

**MELG** (Maximally Equidistributed F$_2$-Linear, Harase-Kimoto 2018)
is a 64-bit family designed to be ME on every resolution by
construction. Uses a lagged-feedback recurrence with two shift
parameters $\sigma_1, \sigma_2$ and a companion `laggedTempering`
output map that XORs in a delayed state word.

This notebook uses MELG-607-64 ($k = 607$, period $2^{607} - 1$).
See [generators/MELGGen.md](../../generators/MELGGen.md).


## Imports


In [ ]:
# stamp:auto-generated
from regpoly.core.generator import Generator
from regpoly.core.combination import Combination
from regpoly.core.transformation import Transformation
from regpoly.analyses.equidistribution_test import (
    EquidistributionTest,
    METHOD_MATRICIAL, METHOD_HARASE,
    METHOD_NOTPRIMITIVE, METHOD_SIMD_NOTPRIMITIVE,
)

INT_MAX = 2**31 - 1


## Construct the generator — _Harase & Kimoto (2018)_


In [ ]:
# Canonical MELG-607-64 parameters (Harase-Kimoto 2018 — Table 1).
gen = Generator.create("MELGGen", L=64,
    w=64, N=10, M=5, r=33,
    sigma1=13, sigma2=35,
    a=0x81F1FD68012348BC)
print(gen.display())

# MELG-style two-reference tempering: y' = (y ^ (y << sigma)) ^
# (state_word_at_lag_L & b).
tempering = Transformation.create("laggedTempering",
    w=64, sigma=30, L=3,
    b=0x66EDC62A6BF8C826)


## Wrap in a `Combination`


In [ ]:
# Wrap the generator + tempering chain in a single-component Combination.
comb = Combination(J=1, Lmax=gen.L)
comb.components[0].add_gen(gen)
comb.components[0].add_trans(tempering)
comb.reset()
print(f"k_g = {comb.k_g}, L = {comb.L}")


## Equidistribution test

MELG is full-period; the **Harase** method is the fastest applicable choice.


In [ ]:
# Build the equidistribution test and run it. We cap `delta` at
# INT_MAX so nothing is rejected — we just want the score.
test = EquidistributionTest(
    L=gen.L,
    delta=[INT_MAX] * (gen.L + 1),
    mse=INT_MAX,
    method=METHOD_HARASE,
)
result = test.run(comb)

print(f"SE (Σ gaps)   = {result.se}")
print(f"verified      = {result.verified}")
print(f"first 10 gaps = {[result.ecart[i] for i in range(1, min(11, gen.L + 1))]}")


## Catalog entry

The published version of this parameter set lives in the REGPOLY catalog under `library_id = "melg607-64"`. To load it programmatically without hard-coding parameters:

```python
from regpoly.library import Catalog
cat = Catalog('docs/library')
cat.load()
_, entry = cat.generator('melg607-64')
# entry.components[0] carries the same params as constructed above
```
